<a href="https://colab.research.google.com/github/mandib96/case-analytics-engineer-2026/blob/dag_airflow/dag_pipeline_vendas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.papermill.operators.papermill import PapermillOperator

In [ ]:
default_args = {
   'owner': 'Amanda Borges',
   'email': ['amandaaborges96@gmail.com'],
   'email_on_failure': True,
   'email_on_retry': False,
   'retries': 2,
   'retry_delay': timedelta(minutes=5)
}

In [ ]:
with DAG(
    dag_id='pipeline_vendas',
    default_args=default_args,
    description='Pipeline de vendas - Lê arquivos no bucket, trata e carrega na camada raw, trusted e analytic.',
    schedule_interval='0 7 * * *',  # Rodo Diariamente às 10h - Horário local
    start_date=datetime(2026, 1, 24),
    end_date=None,
    catchup=False,
    dagrun_timeout=timedelta(minutes=60),
    max_active_runs=1,
    default_args=default_args,
    tags=['vendas', 'data_team', 'franqueados'],
) as dag:

In [ ]:
    ingestao_camada_raw = PapermillOperator(
        task_id="ingestao_camada_raw",
        input_nb="ingestion/create_raw_table.ipynb",
        output_nb="tmp/create_raw_table_{{ds}}.ipynb",
        parameters={"data_execucao": "{{ ds }}"}
    )

    tratamento_camada_trusted_analytics = PapermillOperator(
        task_id="tratamento_camada_trusted_analytics",
        input_nb="transformation/create_trusted_table.ipynb",
        output_nb="tmp/create_trusted_table_{{ds}}.ipynb",
        parameters={"data_execucao": "{{ ds }}"}
    )

    ingestao_camada_raw >> tratamento_camada_trusted_analytics